In [1]:
from google.colab import userdata
from google import genai
from google.genai import types

**Gemini Chatbot Settings & Configuration (Beginner Guide)**

**1. safety_settings**

Controls what kind of content the model is allowed to generate.

- Helps filter harmful, unsafe, or restricted content
- Works using categories like:
  - Hate speech
  - Harassment
  - Dangerous content
- You can define levels like:
  - BLOCK_NONE → Allow everything
  - BLOCK_MEDIUM_AND_ABOVE → Default safe mode
  - BLOCK_ONLY_HIGH → Less strict

Example:

- If you're building a customer support bot, keep safety HIGH
- If you're building a research bot, you may relax it slightly

**2. temperature**

Controls how creative vs predictable the response is.

- Range: 0 → 1 (sometimes up to 2)
- Lower value → More accurate & deterministic
- Higher value → More creative & random

Simple understanding:

- 0.2 → Safe, factual answers (good for SQL, coding)
- 0.7 → Balanced (best for chatbots)
- 1.0+ → Creative (stories, brainstorming)

**3. top_p (Nucleus Sampling)**

Controls how the model selects words based on probability.

- Range: 0 → 1
- Instead of picking from all words, it selects from top probable words

Example:

- top_p = 0.9 → Choose from top 90% probability words
- top_p = 0.3 → Very restricted, less variety

Best practice:

- Usually don’t change both temperature and top_p aggressively
- Common setup:
  - temperature = 0.7
  - top_p = 0.9

**4. types.Part.from_text**

Used to send text input to the model.

- It defines how your input is structured
- Think of it as wrapping your prompt

Example:

types.Part.from_text("Explain machine learning in simple terms")

- You can also send:
  - Images
  - Files
  - Mixed content

In chatbot:

- Every user message → passed using this format

**5. thinking_level**

Controls how deeply the model reasons before answering.

- Not always exposed in basic APIs, but conceptually important

Levels:

- Low → Fast, direct answers
- Medium → Balanced reasoning
- High → Deep thinking (better for complex logic)

Example use:

- Low → Chatbot replies fast
- High → Financial analysis, debugging, ML explanations

Trade-off:

- Higher thinking = slower + more tokens

**6. types.SafetySettings**

This is the actual structure/object format used to define safety rules.

Example:

>types.SafetySetting(
    category="HARM_CATEGORY_HATE_SPEECH",
    threshold="BLOCK_MEDIUM_AND_ABOVE"
)

- You define:
  - Category (what to control)
  - Threshold (how strict)

Common categories:

- Hate speech
- Violence
- Sexual content
- Dangerous activities

**7. max_output_tokens**

Limits how long the response can be.

- 1 token ≈ 1 word (rough approximation)
- Controls:
  - Cost
  - Response length

Example:

- 100 tokens → Short answer
- 500 tokens → Medium explanation
- 1000+ tokens → Detailed output

Best practice:
- Chatbot → 200–500 tokens
- Detailed explanation → 800+

In [2]:

client = genai.Client(api_key = userdata.get('api_key'))
mymodel = "gemini-3.1-flash-lite-preview"

chat = client.chats.create(model=mymodel)

myconfig=types.GenerateContentConfig(
        system_instruction = [
        types.Part.from_text(text="""Professional tone. Always use bullet points to answers""")],
        thinking_config=types.ThinkingConfig(thinking_level="medium"),
        temperature = 1,
        top_p = 0.96,
        max_output_tokens = 1000,
        safety_settings=[
            types.SafetySetting(
                category="HARM_CATEGORY_HATE_SPEECH",
                threshold="BLOCK_ONLY_HIGH",  # Block few
            ),
            types.SafetySetting(
                category="HARM_CATEGORY_DANGEROUS_CONTENT",
                threshold="BLOCK_LOW_AND_ABOVE",  # Block most
            )
        ]
    )

In [3]:
response = client.models.generate_content(
    model=mymodel,
    contents="Hi, My name is Vamshi",
    config=myconfig
)

print(response.text)

* Hello Vamshi, it is a pleasure to meet you.
* Please let me know how I may assist you today with your inquiries or professional requirements.


In [4]:
response = chat.send_message("My Name is Vamshi")
print(response.text)

Hello Vamshi! It's nice to meet you. How are you doing today? Is there anything I can help you with?


In [5]:
# Chat Bot History checking with role

for message in chat.get_history():
    print(f'role - {message.role}',end=": ")
    print(message.parts[0].text)

role - user: My Name is Vamshi
role - model: Hello Vamshi! It's nice to meet you. How are you doing today? Is there anything I can help you with?


***Prompt Engineering***
- Leveraging the Gemini AI large language model for prompt engineering.

**1. Core Prompt Components (Foundation)**

These are the building blocks of every good prompt.

- **Persona (Who the AI should act as)**
- Defines the role or expertise
- Helps control quality + depth of response

Examples:

- “Act as a Digital Marketing Expert”
- “Act as a Data Scientist with 10 years experience”

- **Audience (Who the output is for)**
- Controls language complexity
- Beginner vs Expert difference

Examples:

- “Explain to a beginner”
- “Explain to senior management”

- **Context (Background information)**
- Gives situation clarity
- Without context → generic answers

Examples:

- “I am applying for a Data Analyst role”
- “This is for a startup marketing campaign”

- **Tone (Style of communication)**
- Defines how the output sounds

Types:

- Professional
- Friendly
- Persuasive
- Formal

- **Format (Output structure)**
- Controls how response is presented

Examples:

- Bullet points
- Table
- Step-by-step
- Email format

- **Instruction (What exactly to do)**
- The main task
- Must be clear and specific

Example:

- “Generate 5 resume bullet points”

- **Data (Input content)**
- The actual information given to AI

Example:

- Resume details
- Product description
- Job role info


- **Example (Complete Prompt Structure)**

Marketing Example:

>Persona: Act as a Digital Marketing Expert  
Audience: Target small business owners  
Context: Launching a new organic skincare product  
Tone: Persuasive  
Format: Bullet points  
Instruction: Write 5 ad copies  
Data: Product is chemical-free, affordable, eco-friendly

**2. Prompting Techniques (Core Topics)**

**1. Zero-Shot Prompting**

No examples given

- Direct instruction only
- Model uses its general knowledge
**Example (Job Search):**
>Write a professional resume summary for a Data Analyst.

Simple but sometimes generic

**2. One-Shot Prompting**

Give 1 example

- Helps model understand expected output style
>Example (Marketing):
Example:
Input: Product - Shoes
Output: "Step into comfort with our premium shoes"
Now:
Input: Product - Organic Soap                                            
Output:

**3. Few-Shot Prompting**

Give multiple examples

- Improves accuracy significantly
**Example (Job Search):**
>Example 1:
Skill: Python → "Proficient in Python for data analysis"


>Example 2:
Skill: SQL → "Strong experience in SQL queries"
Now:
Skill: Machine Learning →

**4. Chain of Thought (CoT)**

Ask model to think step-by-step

- Best for:
  - Logic
  - Problem solving
  - Calculations
**Example:**
>Explain how to select the best marketing strategy step by step.

Output will include reasoning steps

**5. Chain of Thought + Self Consistency**

Generate multiple reasoning paths → choose best

- Improves accuracy
- Used in advanced AI systems

**Simple Understanding:**
- Ask same question multiple times
- Compare answers
- Pick most consistent one

Example:

> Solve this problem step by step and verify your answer.

**6. Tree of Thought (ToT)**

Explore multiple possible solutions

- Think like a decision tree
- Best for:
  - Strategy
  - Planning
  - Business decisions

**Example (Marketing Strategy):**
>List 3 different marketing strategies for a startup.
Evaluate pros and cons of each.
Suggest the best one.

**7. Prompt Chaining**

Break task into multiple smaller prompts

- Output of one → input to next
**Example (Job Search Pipeline):**

Step 1:

>Extract key skills from job description

Step 2:

>Match these skills with my resume

Step 3:

>Generate tailored resume bullets

In [6]:
# Building a functioning Loop for chat bot - with continuous flow till a exist criteria met

chat = client.chats.create(model=mymodel)

while True: # this will run for ever till a break criteria is met
  user_input = input('You: ') # we pick what is user provided as input

  if user_input in ['quit','exit','stop']:
    print('Exiting chat....')
    break
  response = chat.send_message(user_input)
  print(f"Model: {response.text}")

You: Hi, do you remember my name
Model: I don't have access to your personal information or a memory of our past conversations unless you've shared your name within this specific chat session. 

If you told me your name earlier in this conversation, I apologize—as an AI, I don't "remember" things across different sessions for privacy and security reasons.

What is your name? I'd be happy to learn it!
You: my self vamshi krishna Marikanti
Model: Nice to meet you, Vamshi Krishna Marikanti! That’s a pleasure to know. How can I help you today?
You: what is my name
Model: Your name is Vamshi Krishna Marikanti.
You: Great
Model: I'm glad we're on the same page! Is there anything specific you'd like to chat about or anything I can help you with today, Vamshi?
You: from i want to test prompt engineering
Model: That sounds like a great plan! Prompt engineering is a fascinating field. Depending on what you want to achieve, there are several ways we can test your skills.

To get started, we could

**Testing the Prompt with real examples**

In [7]:
# Building a functioning Loop for chat bot - with continuous flow till a exist criteria met

chat = client.chats.create(model=mymodel)

while True: # this will run for ever till a break criteria is met
  user_input = input('You: ') # we pick what is user provided as input

  if user_input in ['quit','exit','stop']:
    print('Exiting chat....')
    break
  response = chat.send_message(user_input)
  print(f"Model: {response.text}")

You: What is the Payment value with bill cost of 100$ with 10% Discount and 7% tip Amount
Model: To calculate the final payment, you apply the discount first, then calculate the tip based on the discounted subtotal.

**1. Calculate the Discount:**
*   10% of $100 = $10
*   Subtotal after discount: $100 - $10 = **$90**

**2. Calculate the Tip (on the discounted amount):**
*   7% of $90 = $6.30

**3. Calculate the Final Payment:**
*   $90 (Subtotal) + $6.30 (Tip) = **$96.30**

The total payment value is **$96.30**.
You: What is the payment value with bill cost of 100$ with 10% Discount and 7% tip Amount, Do calculate tip amount on 100$ then discount
Model: If you calculate the tip on the original $100 and then apply the discount, here is the breakdown:

**1. Calculate the Tip (on the original $100):**
*   7% of $100 = **$7.00**

**2. Calculate the Discount (on the $100 bill):**
*   10% of $100 = **$10.00**

**3. Calculate the Final Payment:**
*   $100 (Original Bill) - $10 (Discount) + $

$The End See you Soon$